In [1]:
import multiprocessing
import os
import pathlib

from gensim.models import word2vec
import polars
import numpy
from sklearn import model_selection as sk_model_selection
from sklearn import naive_bayes as sk_nb
from sklearn import preprocessing as sk_pp

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
#INPUT_PATH = EVALUATION_DIR / "output_dataset"
INPUT_PATH = pathlib.Path("/users/lenders/pivot-eval/output_dataset/")

os.environ["EVALUATION_DIR"] = str(EVALUATION_DIR)
os.environ["INPUT_PATH"] = str(INPUT_PATH)

VECTOR_SIZE = 8
WORKERS = multiprocessing.cpu_count()

if WORKERS > 96:
    WORKERS //= 2

PROTOCOLS = ["coap", "coaps", "oscore"]
LINK_LAYER = [""]
BLOCKWISE = [""]
NETWORK_SETUP = ["d1", "d2", "p1", "p2"]
DATA_FORMAT = ["json", "cbor"]
DNS_FORMAT = ["dns_message", "dns_cbor"]

LINK_LAYER_READABLE = {
    "": "eth",
    "-schc": "schc",
}
BLOCKWISE_READABLE = {
    "": "1024",
    "_b16": "16",
}
DNS_FORMAT_READABLE = {
    "dns_message": "application/dns-message",
    "dns_cbor": "application/dns+cbor",
}

# Preparations

In [2]:
list_code(EVALUATION_DIR / "word2vec.py")

#! /usr/bin/env python3
# vim:fenc=utf-8
#
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license

import multiprocessing
import os
import pathlib
import sys
import traceback

from gensim.models import word2vec
import numpy
import polars

EVALUATION_DIR = pathlib.Path.cwd()
INPUT_PATH = pathlib.Path(
    os.environ.get("INPUT_PATH", EVALUATION_DIR / "output_dataset")
)
VECTOR_SIZE = 16
WORKERS = multiprocessing.cpu_count()

if WORKERS > 96:
    WORKERS //= 2

PROTOCOLS = ["coap", "coaps", "oscore"]
LINK_LAYERS = [""]
BLOCKWISE = [""]
NETWORK_SETUPS = ["d1", "d2", "p1", "p2"]
DATA_FORMATS = ["json", "cbor"]
DNS_FORMATS = ["dns_message", "dns_cbor"]


def scenario2vec(scenario):
    try:
        file = INPUT_PATH / f"{scenario}.training.csv.gz" 
        print("Processing", str(file))
        df = polars.read_csv(file, separator=";")
        nibbles, byte_size = df["eth.payload"].str.len_chars().max(), 2

        assert (nibbles % byte_size) == 0
        df = df.with_columns(
            polars.col("eth.payload").map_elements(
                lambda msg: [
                    msg[i : i + byte_size]
                    for i in range(0, nibbles, byte_size)
                ],
                return_dtype=list[str],
            )
        )

        model = word2vec.Word2Vec(
            df["eth.payload"].to_list(),
            workers=WORKERS,
            vector_size=VECTOR_SIZE,
            min_count=1,
            window=3,
            sg=0,
        )

        df_vec = df.with_columns(
            vector=polars.col("eth.payload").map_elements(
                lambda words: numpy.concatenate(
                    [model.wv[word] for word in words]
                ).tolist(),
                return_dtype=polars.List(polars.Float32),
            ),
            label=(polars.col("client.type") != "dns").cast(
                polars.Int8
            ),
        )[["vector", "label"]]
        df_vec.write_parquet(
            INPUT_PATH / f"{scenario}.vector.parquet",
            compression="zstd",
            compression_level=10,
        )
        print("Created", str(INPUT_PATH / f"{scenario}.vector.parquet"))
        del df
        del df_vec
    except Exception:
        print(traceback.format_exc(), file=sys.stderr)
        raise


def main():
    scenarios = []
    for data in DATA_FORMATS:
        for dns in DNS_FORMATS:
            for l2 in LINK_LAYERS:
                for prot in PROTOCOLS:
                    for blk in BLOCKWISE:
                        for stp in NETWORK_SETUPS:
                            scenario = f"{prot}{l2}-{stp}_{data}_{dns}{blk}"
                            file = INPUT_PATH / f"{scenario}.training.csv.gz"
                            vector_file = INPUT_PATH / f"{scenario}.vector.parquet"
                            if file.exists() and not vector_file.exists():
                                scenarios.append(scenario)
                            elif vector_file.exists():
                                print(f"Skipping {file} since {vector_file} exists")

    with multiprocessing.Pool(8 if WORKERS > 8 else WORKERS) as pool:
        pool.map(scenario2vec, scenarios)


if __name__ == "__main__":
    main()

In [ ]:
%%bash

tmux new-session -s "word2vec" -d "source '${EVALUATION_DIR}'/.env/bin/activate && INPUT_PATH='${INPUT_PATH}' '${EVALUATION_DIR}/word2vec.py'"

In [7]:
df_vec = polars.read_parquet(INPUT_PATH / f"{SCENARIO}.vector.parquet")
df_vec = df_vec.cast({"vector": polars.Array(polars.Float32, df_vec["vector"].list.len().max())})
df_vec

vectors,label
"list[array[f32, 8]]",i8
"[[-7.479928, -4.374744, … 2.254304], [8.21677, 1.708123, … -19.191166], … [5.65408, -1.419673, … -3.184898]]",0
"[[-7.479928, -4.374744, … 2.254304], [-7.965573, -3.491143, … 2.332061], … [5.65408, -1.419673, … -3.184898]]",0
"[[-7.479928, -4.374744, … 2.254304], [8.21677, 1.708123, … -19.191166], … [5.65408, -1.419673, … -3.184898]]",1
"[[-7.479928, -4.374744, … 2.254304], [-7.965573, -3.491143, … 2.332061], … [5.65408, -1.419673, … -3.184898]]",1
"[[-7.479928, -4.374744, … 2.254304], [8.21677, 1.708123, … -19.191166], … [5.65408, -1.419673, … -3.184898]]",1
…,…
"[[-7.479928, -4.374744, … 2.254304], [-7.965573, -3.491143, … 2.332061], … [5.65408, -1.419673, … -3.184898]]",1
"[[-7.479928, -4.374744, … 2.254304], [8.21677, 1.708123, … -19.191166], … [5.65408, -1.419673, … -3.184898]]",0
"[[-7.479928, -4.374744, … 2.254304], [-7.965573, -3.491143, … 2.332061], … [5.65408, -1.419673, … -3.184898]]",0


In [40]:
x = df_vec["vector"]
y = df_vec["label"]

x = x.to_numpy()
y = y.to_numpy()

In [41]:
x_train, x_test, y_train, y_test = sk_model_selection.train_test_split(x, y, test_size=0.2)

In [44]:
pandas.DataFrame([0, 1, 2, 3])

,0
0,0
1,1
2,2
3,3


# Naïve Bayes

In [39]:
scaler = sk_pp.MinMaxScaler()
x_train_minmax = scaler.fit_transform(x_train)
x_test_minmax = scaler.transform(x_test)

classifier = sk_nb.MultinomialNB().fit(x_train_minmax, y_train)

ValueError: could not convert string to float: '600cd17000391140fdd86b36eccc00000000000000000002fdd86b36eccc000000000000000000031633cbe50039ac0763449df44f86bc90ff64fd68b1190d5d11871b636cfcc536045abbfd0e1203f3e2890e947da6695cc6f226c4d57f1ad159xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

In [ ]:
y_test_predictions = classifier.predict(x_test_minmax)
conf_matrix = confusion_matrix(y_test, y_test_predictions)
accuracy = accuracy_score(y_test, y_test_predictions)
precision = precision_score(y_test, y_test_predictions)
recall = recall_score(y_test, y_test_predictions)
f1score = f1_score(y_test, y_test_predictions)

print(f"Accuracy = {accuracy:3g}")
print(f"Precision = {precision:3g}")
print(f"Recall = {recall:3g}")
print(f"F1 Score = {f1score:3g}")
conf_matrix